# Codeflaws数据集处理

## 1 依赖导入与全局配置

In [1]:
import os
import json

# =========================================================
# 全局配置：统一管理所有路径与参数
# =========================================================
DATA_DIR             = os.path.join('data')
ERROR_INFO_FILE_PATH = 'codeflaws-defect-detail-info.txt'
OUTPUT_FILE          = 'codeflaws.json'
MIN_CODE_LEN = 35       # buggy_code 或 fixed_code 长度下限（字符数）  
MAX_CODE_LEN = 65535    # buggy_code 或 fixed_code 长度上限（字符数）  

## 2 缺陷类型定义

In [2]:
# 从数据集的论文图表中提取出预定义的缺陷类型

defect_classes = {
    # AST Type: Statement
    "SDIF": {"defect_class": "Delete if, else, else if, for or while", "defect_type": "Control flow"},
    "SIIF": {"defect_class": "Insert if, else, else if, for or while", "defect_type": "Control flow"},
    "SRIF": {"defect_class": "Replace if, else, else if, for or while", "defect_type": "Control flow"},
    "SDRT": {"defect_class": "Delete return", "defect_type": "Control flow"},
    "SIRT": {"defect_class": "Insert return", "defect_type": "Control flow"},
    "SDIB": {"defect_class": "Delete/Insert break or continue", "defect_type": "Control flow"},
    "SDLA": {"defect_class": "Delete assignment", "defect_type": "Data flow"},
    "SISA": {"defect_class": "Insert assignment", "defect_type": "Data flow"},
    "SDFN": {"defect_class": "Delete function call", "defect_type": "Function call"},
    "SISF": {"defect_class": "Insert function call", "defect_type": "Function call"},
    "STYP": {"defect_class": "Replace variable declaration type", "defect_type": "Type"},
    "SMOV": {"defect_class": "Move statement", "defect_type": "Move"},
    "SMVB": {"defect_class": "Move brace up/down", "defect_type": "Move"},

    # AST Type: Operator
    "ORRN": {"defect_class": "Replace relational operator", "defect_type": "Control flow"},
    "OLLN": {"defect_class": "Replace logical operator", "defect_type": "Control flow"},
    "OILN": {"defect_class": "Insert && (tighten condition) or || (loosen condition)", "defect_type": "Control flow"},
    "OEDE": {"defect_class": "Replace = with == or vice versa", "defect_type": "Control flow"},
    "OICD": {"defect_class": "Insert a conditional operator", "defect_type": "Control flow"},
    "OAAN": {"defect_class": "Replace arithmetic operator", "defect_type": "Arithmetic"},
    "OAIS": {"defect_class": "Insert/Delete arithmetic operator", "defect_type": "Arithmetic"},
    "OAID": {"defect_class": "Insert/Delete/Replace ++ or --", "defect_type": "Arithmetic"},
    "OMOP": {"defect_class": "Modify operator precedence", "defect_type": "Arithmetic"},
    "OFFN": {"defect_class": "Alternative function call", "defect_type": "Function call"},
    "OFPF": {"defect_class": "Replace print format", "defect_type": "Function call"},
    "OFPO": {"defect_class": "Modify function parameter order", "defect_type": "Function call"},
    "OIRO": {"defect_class": "Insert/Delete Reference Operator", "defect_type": "Pointer"},
    "OITC": {"defect_class": "Insert type cast operator", "defect_type": "Type"},

    # AST Type: OperanD
    "DCCR": {"defect_class": "Replace constant with variable/constant", "defect_type": "Constant"},
    "DRVA": {"defect_class": "Replace a read variable with a variable/constant", "defect_type": "Variable"},
    "DRWV": {"defect_class": "Replace a write variable with a variable", "defect_type": "Variable"},
    "DMAA": {"defect_class": "Insert/Replace array access", "defect_type": "Array"},
    "DRAC": {"defect_class": "Replace constant of array initialization", "defect_type": "Array"},
    "DCCA": {"defect_class": "Modify array size", "defect_type": "Array"},

    # AST Type: Higher order
    "HDMS": {"defect_class": "Delete multiple non-branch statements", "defect_type": "Non-branch stmt"},
    "HIMS": {"defect_class": "Insert multiple non-branch statements", "defect_type": "Non-branch stmt"},
    "HDIM": {"defect_class": "Delete and insert multiple non-branch statements", "defect_type": "Non-branch stmt"},
    "HBRN": {"defect_class": "Delete/Insert branch statement and non-branch statements", "defect_type": "Branch stmt"},
    "HEXP": {"defect_class": "Delete/Insert/Replace multiple operators and operands", "defect_type": "Expressions"},
    "HCOM": {"defect_class": "Insert/Replace statements and expressions", "defect_type": "Combination"},
    "HOTH": {"defect_class": "Other higher order defect classes", "defect_type": "Others"}
}

print(f"✅ 预定义缺陷类别共 {len(defect_classes)} 种")  


✅ 预定义缺陷类别共 40 种


## 3 工具函数

In [3]:
# ==========
# 工具函数层
# ==========

def load_defect_info(filepath: str) -> dict:
    """
    读取 codeflaws-defect-detail-info.txt，解析每行的缺陷元信息。
    
    Returns:
        dict: { defect_id -> {defect_class, error_type, error_info} }
    """
    defect_dict = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split('\t')
            if len(parts) < 3:
                parts = line.split()
            if len(parts) < 3:
                print(f"[警告] 第 {line_num} 行列数不足3列，已跳过：{line}")
                continue

            defect_id    = parts[0]
            defect_class = parts[1]
            error_type   = parts[2]
            error_info   = parts[3] if len(parts) >= 4 else None

            defect_dict[defect_id] = {
                "defect_class": defect_class,
                "error_type"  : error_type,
                "error_info"  : error_info,
            }
    return defect_dict


def parse_folder_name(folder_name: str) -> tuple[str, str, str, str] | tuple[None, None, None, None]:
    """
    从文件夹名称解析出 contest_id, problem_id, buggy_id, accepted_id。

    文件夹命名规则：<contestid>-<problem>-bug-<buggy_id>-<accepted_id>
    e.g. "1-A-bug-18353198-18353306"
         → ("1", "A", "18353198", "18353306")

    Returns:
        tuple: (contest_id, problem_id, buggy_id, accepted_id)
               解析失败时返回 (None, None, None, None)
    """
    parts = folder_name.split('-')
    if len(parts) < 5 or parts[2] != 'bug':
        return None, None, None, None
    return parts[0], parts[1], parts[3], parts[4]


def read_c_file(filepath: str) -> str:
    """
    读取 C 源文件内容，自动兜底 latin-1 编码。

    Returns:
        str: 文件文本内容，读取失败返回空字符串
    """
    for encoding in ('utf-8', 'latin-1'):
        try:
            with open(filepath, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
        except FileNotFoundError:
            print(f"  [警告] 文件不存在：{filepath}")
            return ""
    print(f"  [警告] 文件编码无法识别：{filepath}")
    return ""


def resolve_bug_type(defect_abbr: str, defect_classes: dict) -> str:
    """
    将缺陷缩写映射为可读的 bug_type 字符串。

    e.g. "SRIF" → "Control flow Defect: Replace if, else, else if, for or while"

    Returns:
        str: 格式化后的 bug_type
    """
    if defect_abbr in defect_classes:
        dc_type = defect_classes[defect_abbr]['defect_type']
        dc_full = defect_classes[defect_abbr]['defect_class']
        return f"{dc_type} Defect: {dc_full}"
    print(f"  [警告] 未知缺陷类别缩写：{defect_abbr}")
    return f"Unknown Defect: {defect_abbr}"


def resolve_error_message(error_type: str, error_info: str | None) -> str:
    """
    将 error_type 与 error_info 拼接为可读的 error_message 字符串。

    e.g. ("WRONG_ANSWER", "~string") → "WRONG_ANSWER: ~string"
         ("RUNTIME_ERROR", None)     → "RUNTIME_ERROR"

    Returns:
        str: 格式化后的 error_message
    """
    return f"{error_type}: {error_info}" if error_info else error_type


def build_record(
    defect_id     : str,
    folder_path   : str,
    contest_id    : str,
    problem_id    : str,
    buggy_id      : str,
    accepted_id   : str,
    defect_info   : dict,
    defect_classes: dict,
    min_code_len  : int = MIN_CODE_LEN,   
    max_code_len  : int = MAX_CODE_LEN,   
) -> dict | None:
    """
    为单条缺陷数据构建 JSON 记录对象。

    Returns:
        dict | None: 成功时返回 JSON 记录，任意关键信息缺失时返回 None
    """
    # ── 1. 查询元信息 ────────────────────────────────────
    if defect_id not in defect_info:
        print(f"[跳过] defect-info 中无记录：{defect_id}")
        return None

    info         = defect_info[defect_id]
    defect_abbr  = info['defect_class']
    error_type   = info['error_type']
    error_info   = info['error_info']

    # ── 2. 解析字段 ──────────────────────────────────────
    bug_type      = resolve_bug_type(defect_abbr, defect_classes)
    error_message = resolve_error_message(error_type, error_info)

    # ── 3. 读取源码文件 ──────────────────────────────────
    buggy_path  = os.path.join(folder_path, f"{contest_id}-{problem_id}-{buggy_id}.c")
    fixed_path  = os.path.join(folder_path, f"{contest_id}-{problem_id}-{accepted_id}.c")

    buggy_code = read_c_file(buggy_path)
    fixed_code = read_c_file(fixed_path)

    if not buggy_code and not fixed_code:
        print(f"[跳过] 无法读取任何源码：{defect_id}")
        return None

    # ── 3.5. 过滤源码长度 ──────────────────────────────────
    if len(buggy_code) < min_code_len or len(buggy_code) > max_code_len:
        print(f"[跳过] 源码长度不符合要求：{defect_id}")
        return None

    if len(fixed_code) < min_code_len or len(fixed_code) > max_code_len:
        print(f"[跳过] 源码长度不符合要求：{defect_id}")
        
    # ── 4. 组装记录 ──────────────────────────────────────
    return {
        "id"           : defect_id,
        "bug_type"     : bug_type,
        "language"     : "C",
        "buggy_code"   : buggy_code,
        "fixed_code"   : fixed_code,
        "error_message": error_message,
    }


## 4 数据集生成

In [4]:
# =========================================================
# 加载 defect-detail-info.txt
# =========================================================
defect_info = load_defect_info(ERROR_INFO_FILE_PATH)

print(f"✅ 成功加载缺陷元信息：{len(defect_info)} 条")

# 预览前3条
for i, (k, v) in enumerate(defect_info.items()):
    if i >= 3: break
    print(f"  {k} → {v}")
    
    
# =========================================================
# 遍历 Codeflaws/data，逐条构建 JSON 记录
# =========================================================
dataset     = []
success_cnt = 0
skip_cnt    = 0

all_folders = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d)) and 'bug' in d
])

print(f"共检测到 {len(all_folders)} 个缺陷文件夹，开始处理...\n")

for folder_name in all_folders:
    folder_path = os.path.join(DATA_DIR, folder_name)

    # Step 1：解析文件夹名
    contest_id, problem_id, buggy_id, accepted_id = parse_folder_name(folder_name)
    if not all([contest_id, problem_id, buggy_id, accepted_id]):
        print(f"[跳过] 文件夹名解析失败：{folder_name}")
        skip_cnt += 1
        continue

    # Step 2：构建单条记录
    record = build_record(
        defect_id      = folder_name,
        folder_path    = folder_path,
        contest_id     = contest_id,
        problem_id     = problem_id,
        buggy_id       = buggy_id,
        accepted_id    = accepted_id,
        defect_info    = defect_info,
        defect_classes = defect_classes,
    )

    if record is None:
        skip_cnt += 1
        continue

    dataset.append(record)
    success_cnt += 1

print(f"\n✅ 处理完成：成功 {success_cnt} 条 | 跳过 {skip_cnt} 条")


# =========================================================
# 保存数据集为 JSON 文件
# =========================================================
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print(f"✅ 数据集已保存至：{OUTPUT_FILE}（共 {len(dataset)} 条记录）")

# ── 预览前2条记录 ─────────────────────────────────────────
print("\n── 数据预览（前2条）──")
for record in dataset[:2]:
    print(json.dumps(record, ensure_ascii=False, indent=2))
    print("-" * 60)


✅ 成功加载缺陷元信息：3903 条
  71-A-bug-18359456-18359477 → {'defect_class': 'DCCR', 'error_type': 'WRONG_ANSWER', 'error_info': 'DIFFOUT~~Loop~If~Printf'}
  4-A-bug-18258406-18258457 → {'defect_class': 'DCCR', 'error_type': 'WRONG_ANSWER', 'error_info': 'DIFFOUT~~If~Else~Printf'}
  231-A-bug-18055428-18055539 → {'defect_class': 'DCCR', 'error_type': 'WRONG_ANSWER', 'error_info': 'DIFFOUT~~If'}
共检测到 3905 个缺陷文件夹，开始处理...

[跳过] defect-info 中无记录：369-C-bug-5354958-5359647
[跳过] defect-info 中无记录：674-E-bug-17842470-17842486

✅ 处理完成：成功 3903 条 | 跳过 2 条
✅ 数据集已保存至：codeflaws.json（共 3903 条记录）

── 数据预览（前2条）──
{
  "id": "1-A-bug-18353198-18353306",
  "bug_type": "Function call Defect: Replace print format",
  "language": "C",
  "buggy_code": "#include<stdio.h>\n#include<math.h>\n\nint main(int argc, char *argv[]){\n\n     double n,m,a;\n     double x;\n\n     scanf(\"%lf %lf %lf\",&n,&m,&a);\n\n\n     x=ceil(n/a)*ceil(m/a);\n\n     printf(\"%d\",(int)x);\n\n     return 0;\n\n}\n",
  "fixed_code": "#include<stdi

## 5 随机抽取n条数据

In [5]:
import json
import random
from pathlib import Path


def sample_dataset(
    json_file: str,
    n: int = 20,
    random_seed: int | None = None,
    verbose: bool = True
) -> list:
    """
    从 DeepFix JSON 数据集中随机抽取 n 条数据并打印显示。

    本函数完全上下文无关，可在 Notebook 的任意 Cell 中独立调用。

    参数:
        json_file    : JSON 文件路径（如 'deepfix_rag_real_pairs.json'）
        n            : 要抽取的数据条数，默认 3
        random_seed  : 随机种子，用于可复现性（默认 None 表示随机）
        verbose      : 是否打印详细信息，默认 True

    返回:
        抽取的数据列表（每条为一个字典）

    异常:
        FileNotFoundError : JSON 文件不存在时抛出
        ValueError        : n 大于数据集总数时抛出
        json.JSONDecodeError : JSON 文件格式不合法时抛出
    """
    import os

    # ---------- 前置校验 ----------
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"JSON 文件不存在: {json_file}")

    # ---------- 加载数据 ----------
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    except json.JSONDecodeError as e:
        raise json.JSONDecodeError(f"JSON 解析失败: {e}", e.doc, e.pos)

    if not isinstance(dataset, list):
        raise ValueError(f"期望 JSON 为列表格式，实际获得: {type(dataset)}")

    total_count = len(dataset)
    if n > total_count:
        raise ValueError(
            f"请求抽取 {n} 条数据，但数据集仅包含 {total_count} 条"
        )

    # ---------- 随机抽样 ----------
    if random_seed is not None:
        random.seed(random_seed)

    sampled_data = random.sample(dataset, n)

    # ---------- 打印输出 ----------
    if verbose:
        print(f"{'='*70}")
        print(f"📊 从 {json_file} 中随机抽取 {n} 条数据（共 {total_count} 条）")
        print(f"{'='*70}\n")

        for idx, entry in enumerate(sampled_data, 1):
            print(f"【样本 {idx}】")
            print(f"  ID          : {entry.get('id', 'N/A')}")
            print(f"  bug_type    : {entry.get('bug_type', 'N/A')}")
            print(f"  language    : {entry.get('language', 'N/A')}")

            # ── 错误代码────────────────────────────────
            buggy_code = entry.get('buggy_code', '')
            buggy_lines = buggy_code.split('\n')
            print(f"\n  【buggy_code】")
            for line in buggy_lines:
                print(f"    {line}")

            # ── 正确代码────────────────────────────────
            fixed_code = entry.get('fixed_code', '')
            fixed_lines = fixed_code.split('\n')
            print(f"\n  【fixed_code】")
            for line in fixed_lines:
                print(f"    {line}")

            # ── 错误信息────────────────────────────────────────
            # error_msg = entry.get('error_message', '')
            # if error_msg:
            #     error_preview = error_msg[:80].replace('\n', '\\n')
            #     print(f"\n  【error_message】")
            #     print(f"    {error_preview}{'...' if len(error_msg) > 80 else ''}")
            
            print(f"\n{'-'*70}\n")

    return sampled_data


def sample_dataset_compact(
    json_file: str,
    n: int = 3,
    random_seed: int | None = None
) -> list:
    """
    轻量版本：从 JSON 中随机抽取 n 条数据，仅返回列表，不打印。
    适合在其他函数中作为子函数调用。

    参数:
        json_file   : JSON 文件路径
        n           : 要抽取的数据条数
        random_seed : 随机种子

    返回:
        抽取的数据列表
    """
    return sample_dataset(json_file, n, random_seed, verbose=False)


# ================================================================
# 快速验证：修改下方参数后直接运行本 Cell 即可查看效果
# ================================================================
if __name__ == '__main__' or True:
    _JSON_FILE    = 'codeflaws.json'  # ← 修改为实际的 JSON 文件路径
    _SAMPLE_SIZE  = 20                               # ← 修改为想要抽取的条数
    _RANDOM_SEED  = 42                              # ← 固定种子可重现结果，设置 None 则随机

    try:
        samples = sample_dataset(
            _JSON_FILE,
            n=_SAMPLE_SIZE,
            random_seed=_RANDOM_SEED
        )
        print(f"\n✅ 成功抽取 {len(samples)} 条数据")

    except FileNotFoundError as e:
        print(f"❌ {e}")
    except ValueError as e:
        print(f"❌ {e}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON 格式错误: {e}")


📊 从 codeflaws.json 中随机抽取 20 条数据（共 3903 条）

【样本 1】
  ID          : 492-A-bug-16962428-16962431
  bug_type    : Non-branch stmt Defect: Delete multiple non-branch statements
  language    : C

  【buggy_code】
    #include <stdio.h>
    #include <stdlib.h>
    
    int main(int argc, char *argv[])
    {
        int n,i,j,x=0,c=0;
    
        scanf("%d",&n);
    
        for(i=1;i<=n;i++)
        {
            for(j=1;j<=i;j++)
               x=x+j;
    
               printf("%d\n",x);
    
            if(x<=n)
                c++;
            else
                break;
        }
    
        printf("%d\n",c);
    
    
    
    
        return 0;
    }
    

  【fixed_code】
    #include <stdio.h>
    #include <stdlib.h>
    
    int main(int argc, char *argv[])
    {
        int n,i,j,x=0,c=0;
    
        scanf("%d",&n);
    
        for(i=1;i<=n;i++)
        {
            for(j=1;j<=i;j++)
               x=x+j;
    
               
    
            if(x<=n)
                c++;
       

## 6 统计问题种类

In [6]:
# =========================================================
# 工具函数：从 defect_id 中提取题目标识 (contest_id + problem_id)
# =========================================================

def extract_problem_id(defect_id: str) -> str | None:
    """
    从缺陷 ID 中提取题目标识。

    命名规则：<contest_id>-<problem_id>-bug-<buggy_id>-<accepted_id>
    e.g. "663-A-bug-17360491-17360583" → "663-A"

    Args:
        defect_id (str): 缺陷 ID 字符串

    Returns:
        str | None: 题目标识，解析失败返回 None
    """
    parts = defect_id.split('-')
    if len(parts) < 5 or parts[2] != 'bug':
        return None
    return f"{parts[0]}-{parts[1]}"


def collect_problem_stats(dataset: list) -> dict:
    """
    遍历数据集，统计每道题目下包含的缺陷记录数。

    Args:
        dataset (list): codeflaws.json 加载后的记录列表

    Returns:
        dict: {
            problem_id (str) -> {
                "count"     : int,        # 该题目下的缺陷数量
                "bug_types" : set[str],   # 涉及的缺陷类型集合
                "defect_ids": list[str]   # 该题目下所有缺陷ID
            }
        }
    """
    stats = {}

    for record in dataset:
        problem_id = extract_problem_id(record['id'])
        if problem_id is None:
            print(f"[警告] 无法解析题目ID：{record['id']}")
            continue

        if problem_id not in stats:
            stats[problem_id] = {
                "count"     : 0,
                "bug_types" : set(),
                "defect_ids": []
            }

        stats[problem_id]["count"]      += 1
        stats[problem_id]["bug_types"].add(record["bug_type"])
        stats[problem_id]["defect_ids"].append(record["id"])

    return stats


# =========================================================
# 加载 codeflaws.json 并执行题目分布统计
# =========================================================

# ── 加载数据集 ────────────────────────────────────────────
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    dataset_loaded = json.load(f)

print(f"✅ 成功加载数据集：共 {len(dataset_loaded)} 条缺陷记录\n")

# ── 执行统计 ──────────────────────────────────────────────
problem_stats = collect_problem_stats(dataset_loaded)

# ── 按缺陷数量降序排列 ────────────────────────────────────
sorted_problems = sorted(
    problem_stats.items(),
    key=lambda x: x[1]['count'],
    reverse=True
)

# ── 打印总览 ──────────────────────────────────────────────
total_problems = len(problem_stats)
total_defects  = sum(v['count'] for v in problem_stats.values())
avg_per_problem = total_defects / total_problems if total_problems else 0

print("=" * 60)
print(f"  题目总数（contest_id-problem_id）：{total_problems} 道")
print(f"  缺陷记录总数                      ：{total_defects} 条")
print(f"  每题平均缺陷数                    ：{avg_per_problem:.2f} 条")
print("=" * 60)

# ── 打印各题目缺陷数分布（Top 10） ────────────────────────
print(f"\n📊 缺陷数 Top 10 题目：")
print(f"  {'题目ID':<15} {'缺陷数':>6}  {'涉及缺陷类型数':>12}")
print(f"  {'-'*15}  {'-'*6}  {'-'*12}")
for problem_id, info in sorted_problems[:10]:
    print(f"  {problem_id:<15} {info['count']:>6}  {len(info['bug_types']):>12}")

# ── 缺陷数分布区间统计 ────────────────────────────────────
print(f"\n📊 题目-缺陷数分布区间：")
brackets = [(1, 1), (2, 3), (4, 6), (7, 10), (11, float('inf'))]
labels   = ['仅 1 条', '2~3 条', '4~6 条', '7~10 条', '11 条以上']

for (lo, hi), label in zip(brackets, labels):
    cnt = sum(
        1 for v in problem_stats.values()
        if lo <= v['count'] <= hi
    )
    bar = '█' * cnt if cnt <= 60 else '█' * 60 + f'(+{cnt-60})'
    print(f"  {label:<8}：{cnt:>4} 道题  {bar}")


✅ 成功加载数据集：共 3903 条缺陷记录

  题目总数（contest_id-problem_id）：1284 道
  缺陷记录总数                      ：3903 条
  每题平均缺陷数                    ：3.04 条

📊 缺陷数 Top 10 题目：
  题目ID               缺陷数       涉及缺陷类型数
  ---------------  ------  ------------
  478-A               20             6
  133-A               19             6
  143-A               18             7
  534-A               17             7
  168-A               16             5
  417-C               16             8
  509-B               16             7
  515-B               15             6
  535-A               15             9
  118-A               14             5

📊 题目-缺陷数分布区间：
  仅 1 条   ： 455 道题  ████████████████████████████████████████████████████████████(+395)
  2~3 条   ： 457 道题  ████████████████████████████████████████████████████████████(+397)
  4~6 条   ： 246 道题  ████████████████████████████████████████████████████████████(+186)
  7~10 条  ：  96 道题  ████████████████████████████████████████████████████████████(+36)
  11 条以上  ：  30